In [1]:
import json
import gdown

import numpy as np
from scipy.io import loadmat

from task_and_baseline import baseline, build_task_helpers


In [3]:
# Download the dataset
url = "https://drive.google.com/file/d/1BBHVSI4KB-B8OX46eN1Nm4ARCeq6Rui4/view?usp=sharing"
downloaded_file = "challenge.mat"
gdown.download(url, downloaded_file, quiet=False)#, fuzzy=True)

data = loadmat("challenge.mat", simplify_cells=True)
tx = data["tx"].astype(np.complex128)
rx = data["rx"].astype(np.complex128)
Fs = float(data["Fs"])
N, _ = tx.shape

tx_n = tx / (np.sqrt(np.mean(np.abs(tx) ** 2, axis=0, keepdims=True)) + 1e-30)
helpers = build_task_helpers(tx_n, Fs, N)


Downloading...
From (original): https://drive.google.com/uc?id=1BBHVSI4KB-B8OX46eN1Nm4ARCeq6Rui4
From (redirected): https://drive.google.com/uc?id=1BBHVSI4KB-B8OX46eN1Nm4ARCeq6Rui4&confirm=t&uuid=b89696b4-922d-4d7f-ba38-1ced5169f2ba
To: C:\Users\user\Documents\Documentation\AI Summer school\SMILES-2026-Signal-main\SMILES-2026-Signal-main\challenge.mat
100%|███████████████████████████████████████████████████████████████████████████████| 393M/393M [02:07<00:00, 3.08MB/s]


!pip install gdown --no-warn-script-location

In [16]:
def rank1_cancel(rx_in, helpers):
    sfilt = helpers["score_filter"]
    band = np.column_stack([sfilt(rx_in[:, c]) for c in range(4)])
    cov = (band.conj().T @ band) / band.shape[0]
    _, vecs = np.linalg.eigh(cov)
    dominant = vecs[:, -1]
    shared = band @ dominant
    shared_power = np.vdot(shared, shared).real + 1e-30
    rx_out = rx_in.copy()
    for c in range(4):
        alpha_c = np.vdot(shared, band[:, c]) / shared_power
        rx_out[:, c] -= alpha_c * shared
    return rx_out


def your_canceller(tx_n, rx):
    fit_tx = helpers["fit_tx_prediction"]

    # Step 1: rank-1 FIRST (remove external jammer)
    rx_hat = rank1_cancel(rx, helpers)

    # Step 2: TX cancellation on the cleaner signal
    rx_hat = rx_hat - fit_tx(rx_hat)

    return rx_hat

print("\n=== Baseline ===")
baseline_reds, baseline_avg = helpers["score"](
    rx, baseline(tx_n, rx, helpers["fit_tx_prediction"]), label="baseline"
)

print("=== Your Solution ===")
yours_reds, yours_avg = helpers["score"](rx, your_canceller(tx_n, rx), label="yours")

results = {
    "baseline": {
        "per_channel_db": baseline_reds,
        "average_db": baseline_avg,
    },
    "yours": {
        "per_channel_db": yours_reds,
        "average_db": yours_avg,
    },
}

with open("results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

print(f"\nBaseline : {baseline_avg:.2f} dB")
print(f"Yours    : {yours_avg:.2f} dB")
print(f"Gain     : +{yours_avg - baseline_avg:.2f} dB")


=== Baseline ===
  ch0: 3.98 dB
  ch1: 4.86 dB
  ch2: 3.49 dB
  ch3: 3.74 dB
  Metric [baseline]: 4.02 dB

=== Your Solution ===
  ch0: 9.41 dB
  ch1: 7.15 dB
  ch2: 8.63 dB
  ch3: 7.29 dB
  Metric [yours]: 8.12 dB


Baseline : 4.02 dB
Yours    : 8.12 dB
Gain     : +4.10 dB
